# MultiRate — train all warehouse history and score the latest date

Run All trains the current warehouse equity model through the latest stored equity-price date, including the current partial year. Training and prediction use `quant-orchestrator`; features and labels come from `quant-warehouse`. The model uses the same architecture, supervised heads, and reconstruction objectives as the completed equities-only research run.

The resulting scores describe the last stored date after fitting on all available history; they are not an out-of-sample backtest. Symbols without a price on that common date are reported and excluded from the leaderboard.

Options use a simple rule: expiry nearest **60 calendar days**, then strike nearest the same-date equity close. Long signals select calls; short signals select puts. Expiry-distance ties prefer the later expiry, and strike-distance ties prefer the lower strike. No option model or LLM ranks contracts. Order plans remain optional and require manual review/submission in the existing application.

In [ ]:
# Edit settings here, then Run All.
import os
import sys
from pathlib import Path
from datetime import datetime, timezone

MIN_MARKET_CAP = 10_000_000_000
EPOCHS = 1
BATCH_SIZE = 64
D_MODEL = 64
NUM_HEADS = 4
LAYERS = 2
RECONSTRUCTION_WEIGHT = 0.1
CHECKPOINT_EVERY_BATCHES = 10
PROGRESS_UPDATES_PER_EPOCH = 10
SEED = 0
DEVICE = "cuda"
POLARS_THREADS = 8
OMP_THREADS = 4

TOP_K = 20
MIN_LONG_SCORE = 0.50
OPTION_TENOR_DAYS = 60
OPTION_STRATEGY_ALLOCATION = 100_000.0
BUILD_ORDER_PLANS = False  # Enable to read existing Alpaca accounts and prepare reviewable plans.
ALPACA_LIVE_OPTION_DISCOUNT_PCT = float(os.getenv("TRADING_APP_V2_ALPACA_LIVE_OPTION_DISCOUNT_PCT", "90.0"))

os.environ["POLARS_MAX_THREADS"] = str(POLARS_THREADS)
os.environ["OMP_NUM_THREADS"] = str(OMP_THREADS)
roots = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "optimal_trader"]
REPO_ROOT = next(root for root in roots if (root / "app/trading_app_v2_runtime.py").exists())
ORCHESTRATOR_ROOT = Path(os.getenv("QUANT_ORCHESTRATOR_ROOT", str(REPO_ROOT.parent / "quant-orchestrator")))
sys.path.insert(0, str(REPO_ROOT))
if ORCHESTRATOR_ROOT.exists():
    sys.path.insert(0, str(ORCHESTRATOR_ROOT))
UNIVERSE_TAG = f"{MIN_MARKET_CAP / 1e12:g}T" if MIN_MARKET_CAP >= 1e12 else f"{MIN_MARKET_CAP / 1e9:g}B"
RUN_ID = datetime.now(timezone.utc).strftime("latest_%Y%m%dT%H%M%S_%fZ")
RUN_DIR = ORCHESTRATOR_ROOT / "artifacts/multirate_recovery" / UNIVERSE_TAG / RUN_ID
LIVE_DIR = REPO_ROOT / "artifacts/trading_app_v2/multirate_live" / UNIVERSE_TAG / RUN_ID

In [ ]:
import json
import pandas as pd
import torch
from dotenv import load_dotenv
from quant_warehouse.warehouse.api import Warehouse
from quant_orchestrator.research_tools.warehouse_live import train_latest_warehouse_model
from app.trading_app_v2_runtime import (
    build_alpaca_equity_orders,
    build_latest_equity_leaderboard,
    build_ranked_alpaca_option_orders,
    load_multirate_strategy_scores,
    select_atm_options,
    save_live_artifacts,
    write_streamlit_leaderboard_app,
)

load_dotenv(REPO_ROOT / ".env", override=False)
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("This run requires a CUDA notebook kernel.")
warehouse = Warehouse()
LIVE_DIR.mkdir(parents=True, exist_ok=False)
display({"training_output": str(RUN_DIR), "live_output": str(LIVE_DIR),
         "min_market_cap": MIN_MARKET_CAP, "options_training": False,
         "score_date_policy": "latest finite equity close stored in the selected warehouse universe"})

## Train and predict

This calls the shared optimized warehouse workflow: full history from `1900-01-01`, four preparation workers, batched annual memory, all equity supervised objectives, masked reconstruction and next-token prediction. It starts fresh weights, saves a checkpoint, then scores only the latest warehouse date using that year's available context. It does not run historical backtests, refresh market data, or train/score option models.

In [ ]:
training_result = train_latest_warehouse_model(
    RUN_DIR, min_market_cap=MIN_MARKET_CAP, epochs=EPOCHS, batch_size=BATCH_SIZE,
    d_model=D_MODEL, num_heads=NUM_HEADS, layers=LAYERS, device=DEVICE, seed=SEED,
    reconstruction_weight=RECONSTRUCTION_WEIGHT,
    checkpoint_every_batches=CHECKPOINT_EVERY_BATCHES,
    progress_updates_per_epoch=PROGRESS_UPDATES_PER_EPOCH,
    warehouse=warehouse,
)
score_date = training_result["score_date"]
CHECKPOINT_PATH = Path(training_result["checkpoint"])
display(training_result)

In [ ]:
strategy_scores = load_multirate_strategy_scores(Path(training_result["prediction_path"]))
latest_prices = pd.read_parquet(training_result["prices_path"])
assert set(pd.to_datetime(strategy_scores["date"]).dt.strftime("%Y-%m-%d")) == {score_date}
price_map = latest_prices.set_index("symbol")["close"].to_dict()
leaderboard = build_latest_equity_leaderboard(
    strategy_scores, top_k=TOP_K, min_long_score=MIN_LONG_SCORE, price_map=price_map,
)
display(leaderboard.head(TOP_K + 5))
print({"score_date": score_date, "scored_equities": len(strategy_scores),
       "missing_latest_prices": training_result["missing_latest_prices"]})

## Select ATM options near 60 DTE

Read the complete locally stored chain on the scoring date for ranked eligible equities. Choose only the signal's option side. If that date's chain or required side is missing, record the reason and try the next ranked equity; do not silently use an older chain. Selection does not optimize option predictions, liquidity, or historical outcomes.

In [ ]:
option_rankings, option_selection_audit = select_atm_options(
    leaderboard, score_date=score_date, target_dte=OPTION_TENOR_DAYS,
    top_k=TOP_K, warehouse=warehouse,
)
selected_symbols = option_rankings["symbol"].tolist()
option_leaderboard = leaderboard.loc[leaderboard["symbol"].isin(selected_symbols)].copy()
option_leaderboard["selected"] = True
option_selection_audit.to_csv(LIVE_DIR / "option_selection_audit.csv", index=False)
display(option_rankings)
display(option_selection_audit)

In [ ]:
paper_order_plans = {}
if BUILD_ORDER_PLANS:
    paper_order_plans["alpaca_equity_paper"] = build_alpaca_equity_orders(
        leaderboard=leaderboard, account_prefix="EQUITY", gross_exposure=0.95,
    )
    for name, live in [("alpaca_option_paper", False), ("alpaca_option_live", True)]:
        paper_order_plans[name] = build_ranked_alpaca_option_orders(
            option_rankings=option_rankings, decisions=option_leaderboard[["symbol", "direction"]],
            account_prefix="OPTION", strategy_allocation=OPTION_STRATEGY_ALLOCATION,
            max_underlyings=TOP_K, live=live,
            discount_pct=ALPACA_LIVE_OPTION_DISCOUNT_PCT if live else 0.0,
        )
    for name, frame in paper_order_plans.items():
        print(name, len(frame))
        display(frame)
else:
    print("Predictions and option selections are ready. Order-plan generation is disabled.")

In [ ]:
saved = save_live_artifacts(
    live_dir=LIVE_DIR, leaderboard=leaderboard, symbol_scores=strategy_scores,
    option_ml_rankings=option_rankings, orders=paper_order_plans,
)
streamlit_app = write_streamlit_leaderboard_app(
    live_dir=LIVE_DIR, leaderboard=leaderboard, symbol_scores=strategy_scores,
    option_ml_rankings=option_rankings, orders=paper_order_plans,
)
(LIVE_DIR / "model_run.json").write_text(json.dumps(training_result, indent=2))
print({"checkpoint": str(CHECKPOINT_PATH), "score_date": score_date,
       "saved": saved, "streamlit_app": str(streamlit_app)})
print(f"Run: streamlit run {streamlit_app} --server.address 127.0.0.1")